In [ ]:
import math
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.style as style
import numpy as np
import pandas as pd

import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway
import datetime
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
# 데이터 시각화
import matplotlib.pyplot as plt
import matplotlib

# 맑은 고딕 적용
matplotlib.rc("font", family = "AppleGothic")
# 음수 표시
matplotlib.rc("axes", unicode_minus = False)

# items 데이터셋

In [14]:
df_items = pd.read_csv("C반과제-유통-새벽배송매출증대/on_items.csv", encoding="cp949")
df.head()

,ItemLargeCode,ItemLargeName,ItemMiddleCode,ItemMiddleName,ItemSmallCode,ItemSmallName,ItemCode,ItemName,PriceYear,PriceMin,PriceMax
0,L1,가공식품,M11,곡물,S0080,국수,L1-M11-S0080-1001,(식품)샘표 김치국수 101g 10입,2023,15840,16130
1,L1,가공식품,M11,곡물,S0080,국수,L1-M11-S0080-1001,(식품)샘표 김치국수 101g 10입,2024,17030,17340
2,L1,가공식품,M11,곡물,S0080,국수,L1-M11-S0080-1001,(식품)샘표 김치국수 101g 10입,2025,17380,18640
3,L1,가공식품,M11,곡물,S0080,국수,L1-M11-S0080-1002,2.1kg 6배 메밀 Bestco 희석용 소바 국수장국,2024,14160,15350
4,L1,가공식품,M11,곡물,S0080,국수,L1-M11-S0080-1002,2.1kg 6배 메밀 Bestco 희석용 소바 국수장국,2025,15060,16160


In [11]:
df_items.shape

(10054, 11)

In [18]:
df_items.isnull().sum()

ItemLargeCode     0
ItemLargeName     0
ItemMiddleCode    0
ItemMiddleName    0
ItemSmallCode     0
ItemSmallName     0
ItemCode          0
ItemName          0
PriceYear         0
PriceMin          0
PriceMax          0
dtype: int64

In [34]:
print("=== df_items 고유값 개수 ===")
print(df_items.nunique())

=== df_items 고유값 개수 ===
ItemLargeCode        4
ItemLargeName        4
ItemMiddleCode      14
ItemMiddleName      14
ItemSmallCode       58
ItemSmallName       59
ItemCode          3990
ItemName          3989
PriceYear            3
PriceMin          2917
PriceMax          3070
dtype: int64


# orders 데이터셋

In [40]:
df_orders = pd.read_csv("C반과제-유통-새벽배송매출증대/on_orders.csv", encoding="cp949")
df_orders.head()

,idUser,idOrder,OrderDT,ItemCode,Price,DeliveryDT
0,U10001,U10001-O2023-1002,06JAN2023:17:08:51,L4-M17-S0530-1024,33310,07JAN2023:06:24:00
1,U10001,U10001-O2023-1002,06JAN2023:17:08:51,L1-M21-S0540-1082,3780,07JAN2023:06:24:00
2,U10001,U10001-O2023-1002,06JAN2023:17:08:51,L1-M15-S0140-1311,22520,07JAN2023:06:24:00
3,U10001,U10001-O2023-1002,06JAN2023:17:08:51,L4-M12-S0350-1035,21630,07JAN2023:06:24:00
4,U10001,U10001-O2023-1003,13JAN2023:16:50:14,L4-M12-S0640-1057,11700,14JAN2023:06:28:00


In [38]:
df_orders.shape

(855365, 6)

In [39]:
df_orders.isnull().sum()

idUser        0
idOrder       0
OrderDT       0
ItemCode      0
Price         0
DeliveryDT    0
dtype: int64

In [35]:
print("\n=== df_orders 고유값 개수 ===")
print(df_orders.nunique())


=== df_orders 고유값 개수 ===
idUser          3000
idOrder       171431
OrderDT       171351
ItemCode        3989
Price           3986
DeliveryDT    106844
dtype: int64


# users 데이터셋

In [41]:
df_users = pd.read_csv("C반과제-유통-새벽배송매출증대/on_users.csv", encoding="cp949")
df_users.head()

,idUser,Gender,Age,FamilyCount,MemberYN
0,U10001,여성,26,2,Y
1,U10002,남성,61,2,Y
2,U10003,여성,34,2,Y
3,U10004,남성,26,1,N
4,U10005,여성,33,3,Y


In [26]:
df_users.shape

(3000, 5)

In [27]:
df_users.isnull().sum()

idUser         0
Gender         0
Age            0
FamilyCount    0
MemberYN       0
dtype: int64

In [36]:
print("\n=== df_users 고유값 개수 ===")
print(df_users.nunique())


=== df_users 고유값 개수 ===
idUser         3000
Gender            2
Age              49
FamilyCount       4
MemberYN          2
dtype: int64


# 데이터 병합

In [ ]:
df_orders['PriceYear'] = pd.to_datetime(df_orders['OrderDT'], format='%d%b%Y:%H:%M:%S', errors='coerce').dt.year

df = df_orders.merge(df_users, on='idUser', how='left')

df = df.merge(df_items, on=['ItemCode', 'PriceYear'], how='left')

print("=== 통합 데이터셋 정보 ===")
print(f"Shape: {df.shape}")
print(f"\nPriceYear 누락값: {df['PriceYear'].isnull().sum()}")
print(f"\n고유값 개수:")
print(df.nunique())
print(f"\n누락값 개수:")
print(df.isnull().sum())
print(f"\n샘플 데이터 (처음 5행):")
df.head()

=== 통합 데이터셋 정보 ===
Shape: (857037, 20)

PriceYear 누락값: 1023

고유값 개수:
idUser              3000
idOrder           171431
OrderDT           171351
ItemCode            3989
Price               3986
DeliveryDT        106844
PriceYear              3
Gender                 2
Age                   49
FamilyCount            4
MemberYN               2
ItemLargeCode          4
ItemLargeName          4
ItemMiddleCode        14
ItemMiddleName        14
ItemSmallCode         58
ItemSmallName         59
ItemName            3988
PriceMin            2917
PriceMax            3070
dtype: int64

누락값 개수:
idUser               0
idOrder              0
OrderDT              0
ItemCode             0
Price                0
DeliveryDT           0
PriceYear         1023
Gender               0
Age                  0
FamilyCount          0
MemberYN             0
ItemLargeCode     1460
ItemLargeName     1460
ItemMiddleCode    1460
ItemMiddleName    1460
ItemSmallCode     1460
ItemSmallName     1460
ItemName          

,idUser,idOrder,OrderDT,ItemCode,Price,DeliveryDT,PriceYear,Gender,Age,FamilyCount,MemberYN,ItemLargeCode,ItemLargeName,ItemMiddleCode,ItemMiddleName,ItemSmallCode,ItemSmallName,ItemName,PriceMin,PriceMax
0,U10001,U10001-O2023-1002,06JAN2023:17:08:51,L4-M17-S0530-1024,33310,07JAN2023:06:24:00,2023.0,여성,26,2,Y,L4,신선식품,M17,수산,S0530,전복,완도 활전복 1kg 중 22-25미,33160.0,37070.0
1,U10001,U10001-O2023-1002,06JAN2023:17:08:51,L1-M21-S0540-1082,3780,07JAN2023:06:24:00,2023.0,여성,26,2,Y,L1,가공식품,M21,즉석,S0540,즉석,동원 양반 차돌된장찌개 (460G),3690.0,3970.0
2,U10001,U10001-O2023-1002,06JAN2023:17:08:51,L1-M15-S0140-1311,22520,07JAN2023:06:24:00,2023.0,여성,26,2,Y,L1,가공식품,M15,냉동,S0140,냉동,오뚜기 듬뿍 새우볶음밥450g (2인분) x 5봉지 /,22150.0,23150.0
3,U10001,U10001-O2023-1002,06JAN2023:17:08:51,L4-M12-S0350-1035,21630,07JAN2023:06:24:00,2023.0,여성,26,2,Y,L4,신선식품,M12,과일,S0350,사과,[산지직송] 새콤달콤 부사 사과 5kg (13과내),20810.0,23030.0
4,U10001,U10001-O2023-1003,13JAN2023:16:50:14,L4-M12-S0640-1057,11700,14JAN2023:06:28:00,2023.0,여성,26,2,Y,L4,신선식품,M12,과일,S0640,토마토,스테비아 방울 토마토 라루 토망고 1kg,11640.0,13020.0


In [ ]:
df = df.dropna()

In [46]:
df.isnull().sum()

idUser            0
idOrder           0
OrderDT           0
ItemCode          0
Price             0
DeliveryDT        0
PriceYear         0
Gender            0
Age               0
FamilyCount       0
MemberYN          0
ItemLargeCode     0
ItemLargeName     0
ItemMiddleCode    0
ItemMiddleName    0
ItemSmallCode     0
ItemSmallName     0
ItemName          0
PriceMin          0
PriceMax          0
dtype: int64

In [47]:
df.shape

(855577, 20)